# 第17章 大模型与金融前沿 —— 配套代码

对应正文 `book/part4/17-llm-finance.md`。

> **离线运行说明**：本 notebook 所有 cell 均可在**无 API Key 的离线环境**下运行。
> 真实 LLM API 调用示例用 `try/except` 保护，无 Key 时会优雅跳过。

## 演示内容

1. 环境初始化
2. LLM 核心概念可视化（温度、上下文窗口）
3. 金融知识库构建（硬编码片段）
4. TF-IDF 向量化（模拟嵌入）
5. RAG 检索演示：Top-k 相关片段 + 热力图
6. 结构化提示模板构造
7. 模拟 LLM 输出解析（JSON）
8. 情感分析提示模板（Few-shot）
9. 思维链提示示例
10. 真实 LLM API 调用示例（离线保护）
11. RAG 检索质量评估（MRR / Hit@k）
12. 幻觉风险演示与验证策略
13. 综合流水线演示
14. 习题参考解答

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1: 环境初始化
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, json, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
try:
    from fds import set_chinese_font; set_chinese_font()
except ImportError:
    plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
SEED = 42; np.random.seed(SEED)
print('环境初始化完成')
import sklearn
print(f'numpy {np.__version__}  pandas {pd.__version__}  sklearn {sklearn.__version__}')

## 17.1 LLM 核心概念可视化

- **Token**：文本的最小处理单元
- **温度（Temperature）**：控制输出随机性（0=确定，2=随机）
- **上下文窗口**：一次能处理的最大 token 数

In [ ]:
# Cell 2: 温度 + 上下文窗口可视化
tokens = ['增长', '下滑', '持平', '波动', '改善']
logits = np.array([2.5, 1.8, 1.2, 0.9, 0.3])
def softmax_T(lg, T):
    s = lg / T; e = np.exp(s - s.max()); return e / e.sum()
temperatures = [0.1, 0.5, 1.0, 2.0]
t_colors = ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax1 = axes[0]; x = np.arange(len(tokens)); w = 0.2
for i, (T, c) in enumerate(zip(temperatures, t_colors)):
    ax1.bar(x + i*w, softmax_T(logits, T), width=w, color=c, alpha=0.8, label=f'T={T}')
ax1.set_xticks(x + w*1.5); ax1.set_xticklabels(tokens, fontsize=11)
ax1.set_ylabel('采样概率'); ax1.set_title('温度对 Token 采样概率的影响')
ax1.legend(title='Temperature'); ax1.set_ylim(0, 1)
ax2 = axes[1]
models = ['GPT-3\n(2020)', 'GPT-3.5\n(2022)', 'GPT-4\n(2023)', 'Qwen2.5\n(2024)', 'Kimi\n(2024)']
ctx_k = [4, 16, 128, 1000, 1000]
b_cols = ['#aec7e8', '#7fbfff', '#1f77b4', '#d62728', '#ff7f0e']
bars = ax2.bar(models, ctx_k, color=b_cols, alpha=0.85, edgecolor='white')
ax2.set_ylabel('上下文窗口（K tokens）'); ax2.set_title('主流 LLM 上下文窗口大小对比')
ax2.set_ylim(0, 1200)
ax2.axhline(200, color='gray', linestyle='--', lw=1.5, alpha=0.7, label='典型年报 ~200K token')
ax2.legend(fontsize=9)
for bar, val in zip(bars, ctx_k):
    ax2.text(bar.get_x()+bar.get_width()/2, val+20, f'{val}K',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.suptitle('LLM 核心参数可视化', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'T=0.1 最高概率={softmax_T(logits, 0.1).max():.3f}  T=2.0 最高概率={softmax_T(logits, 2.0).max():.3f}')
print('金融场景建议：信息抽取 T=0~0.2，摘要生成 T=0.3~0.7')

## 17.2 金融知识库构建

硬编码一组金融知识片段，模拟已经过分块（chunking）处理的文档。

In [ ]:
# Cell 3: 构建金融知识库（硬编码片段，模拟 RAG 索引阶段）
knowledge_base = [
    {'id': 'KB001', 'source': '贵州茅台2023年年报',
     'text': '贵州茅台2023年实现营业收入1476.94亿元，同比增长18.04%；归母净利润747.34亿元，同比增长19.16%。茅台酒收入1278.54亿元，占总收入86.6%；系列酒收入198.40亿元，同比增长42.03%。公司全年分红总额达238.45亿元，每股分红18.96元。'},
    {'id': 'KB002', 'source': '中国人民银行货币政策报告（2024Q1）',
     'text': '2024年一季度，货币政策保持稳健，LPR 1年期3.45%，5年期以上3.95%。央行全面降准0.5个百分点，释放长期资金约1万亿元。M2同比增长8.3%。'},
    {'id': 'KB003', 'source': '沪深交易所科创板上市规则',
     'text': '科创板上市标准之一：市值不低于10亿元，最近两年净利润均为正且累计净利润不低于5000万元。或市值不低于15亿元，最近一年营业收入不低于2亿元。科创属性：研发投入占营收比例原则上不低于5%。'},
    {'id': 'KB004', 'source': '宁德时代2023年年报',
     'text': '宁德时代2023年营业收入4009.17亿元，同比增长22.01%；归母净利润441.21亿元，同比增长43.58%。动力电池出货量254GWh，储能电池出货量69GWh。公司研发费用183.49亿元，研发人员近2万人，海外收入占比提升至33.2%。'},
    {'id': 'KB005', 'source': 'A股IPO注册制改革说明',
     'text': '2023年全面推行注册制后，A股IPO审核由核准制转为注册制。主要变化：审核主体由证监会转为交易所，证监会负责注册；审核重点从盈利能力转向信息披露质量；上市时间缩短至平均12-18个月。'},
    {'id': 'KB006', 'source': '银监会巴塞尔III实施指引',
     'text': '商业银行资本充足率监管要求：核心一级资本充足率不低于5%，一级资本充足率不低于6%，资本充足率不低于8%。流动性覆盖率（LCR）不低于100%，净稳定资金比例（NSFR）不低于100%。'},
    {'id': 'KB007', 'source': '比亚迪2023年年报',
     'text': '比亚迪2023年营业收入6023.15亿元，同比增长42.04%；归母净利润300.41亿元，同比增长80.72%。新能源汽车销量302.44万辆，同比增长62.3%，连续第二年蝉联全球新能源汽车销冠。研发投入395.75亿元，同比增长97.2%。'},
    {'id': 'KB008', 'source': '证监会投资者适当性管理办法',
     'text': '经营机构向投资者销售产品或提供服务，须履行适当性义务。投资者分类：普通投资者和专业投资者。产品分级：从R1（低风险）到R5（高风险）共5个风险等级。'},
    {'id': 'KB009', 'source': 'A股市场估值参考（2024）',
     'text': '2024年3月末，沪深300指数整体市盈率（TTM）约12.5倍，处于历史中低水平。分行业：银行板块约4.5倍，消费白酒约20倍，新能源约15倍，科技成长约30倍。标普500约25倍，港股恒生指数约9倍，日经225约20倍。'},
    {'id': 'KB010', 'source': 'DeepSeek技术报告（2025）',
     'text': 'DeepSeek-V3采用混合专家（MoE）架构，总参数量671B，激活参数37B。训练成本约557万美元，远低于同级别闭源模型。在代码、数学、中文理解等基准测试上与GPT-4o相当甚至超越。开源发布，支持私有化部署，在金融机构中获得广泛关注。'},
]

corpus = [doc['text'] for doc in knowledge_base]
doc_ids = [doc['id'] for doc in knowledge_base]
doc_sources = [doc['source'] for doc in knowledge_base]

print(f'金融知识库共 {len(knowledge_base)} 条文档片段')
for doc in knowledge_base[:3]:
    print(f'  [{doc["id"]}] {doc["source"]}')
    print(f'    {doc["text"][:60]}...')

## 17.3 TF-IDF 向量化（模拟 Embedding）

用 **TF-IDF** 作为深度嵌入模型的轻量替代，完全离线、无需 GPU。

相似度用**余弦相似度**衡量，范围 [-1, 1]，值越大越相似。

In [ ]:
# Cell 4: TF-IDF 向量化 + 余弦相似度检索（模拟 RAG 索引+检索）
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3),
                             max_features=5000, sublinear_tf=True)
doc_vectors = vectorizer.fit_transform(corpus)
print(f'TF-IDF 矩阵形状：{doc_vectors.shape}')
print(f'  稀疏度：{1-doc_vectors.nnz/(doc_vectors.shape[0]*doc_vectors.shape[1]):.2%}')

def retrieve(query, top_k=3, verbose=True):
    qv = vectorizer.transform([query])
    sc = cosine_similarity(qv, doc_vectors)[0]
    idx = sc.argsort()[::-1][:top_k]
    results = [{'rank': r+1, 'id': doc_ids[i], 'source': doc_sources[i],
                'score': float(sc[i]), 'text': corpus[i]} for r, i in enumerate(idx)]
    if verbose:
        print(f'Query: {query}')
        for r in results:
            print(f'  Rank {r["rank"]} [{r["id"]}] sim={r["score"]:.4f} {r["source"]}')
            print(f'    {r["text"][:70]}...')
    return results

_ = retrieve('茅台的营业收入和净利润增速是多少？', top_k=3)

In [ ]:
# Cell 5: 多问题检索演示 + 相似度热力图
test_queries = [
    '科创板上市需要满足哪些财务条件？',
    '银行资本充足率的监管要求是什么？',
    '新能源汽车销量冠军是哪家公司？',
    'DeepSeek模型有什么特点？',
    'A股市场的市盈率水平如何？',
]
print('=== 多查询检索演示 ===')
for q in test_queries:
    r = retrieve(q, top_k=1, verbose=False)[0]
    print(f'Q: {q[:30]} -> [{r["id"]}] sim={r["score"]:.4f}')

fig, ax = plt.subplots(figsize=(14, 5))
sim_matrix = cosine_similarity(vectorizer.transform(test_queries), doc_vectors)
im = ax.imshow(sim_matrix, cmap='YlOrRd', aspect='auto', vmin=0)
ax.set_yticks(range(len(test_queries)))
ax.set_yticklabels([q[:16]+'...' for q in test_queries], fontsize=9)
ax.set_xticks(range(len(doc_ids)))
ax.set_xticklabels(doc_ids, rotation=45, ha='right', fontsize=9)
ax.set_title('查询 x 知识库片段 TF-IDF 余弦相似度热力图（RAG 检索可视化）')
for i in range(len(test_queries)):
    for j in range(len(doc_ids)):
        if sim_matrix[i, j] > 0.1:
            ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                    fontsize=7.5, color='black' if sim_matrix[i,j] < 0.3 else 'white')
plt.colorbar(im, ax=ax, shrink=0.8, label='余弦相似度')
plt.tight_layout(); plt.show()

## 17.4 结构化提示模板构造

在 RAG 系统中，检索到相关片段后，需要将其组装成**提示（Prompt）**发送给 LLM。
提示的质量直接决定输出质量。

In [ ]:
# Cell 6: 结构化提示模板构造
def build_rag_prompt(query, retrieved_docs, task='qa'):
    context = '\n\n'.join([f'[{d["id"]}] {d["source"]}: {d["text"]}' for d in retrieved_docs])
    if task == 'qa':
        return (
            '你是一位专业的金融分析师。请基于以下参考资料回答用户问题。\n'
            '要求：1.只使用参考资料信息 2.包含具体数字 3.注明来源 4.信息不足请说明\n\n'
            f'=== 参考资料 ===\n{context}\n\n'
            f'=== 用户问题 ===\n{query}\n\n=== 回答 ==='
        )
    else:
        return (
            '从参考资料中提取结构化信息，按JSON格式输出：\n'
            '{公司名称, 报告年度, 营业收入（亿元）, 收入增速, 净利润（亿元）, 净利润增速}\n\n'
            f'参考资料：\n{context}\n\n问题：{query}\n\nJSON输出：'
        )

query = '茅台2023年的营收和净利润分别是多少？增速如何？'
retrieved = retrieve(query, top_k=2, verbose=False)
prompt_qa = build_rag_prompt(query, retrieved, task='qa')
print('=== RAG 问答提示（前500字符）===')
print(prompt_qa[:500])
print(f'...\n提示总长度：{len(prompt_qa)} 字符，约 {len(prompt_qa)//2} tokens（估算）')

## 17.5 模拟 LLM 输出解析

LLM 的输出需要经过解析才能入库或展示。
本节演示如何解析 JSON 输出，以及处理常见解析错误。

In [ ]:
# Cell 7: 模拟 LLM 输出解析（JSON 结构化输出）
sim_maotai = json.dumps({'公司名称': '贵州茅台', '报告年度': '2023',
    '营业收入（亿元）': 1476.94, '收入增速': '18.04%',
    '净利润（亿元）': 747.34, '净利润增速': '19.16%',
    '核心业务': ['茅台酒', '系列酒'], '数据来源': 'KB001'}, ensure_ascii=False, indent=2)
sim_byd = '根据年报：\n' + json.dumps({'公司名称': '比亚迪', '报告年度': '2023',
    '营业收入（亿元）': 6023.15, '收入增速': '42.04%',
    '净利润（亿元）': 300.41, '净利润增速': '80.72%',
    '核心业务': ['新能源汽车', '动力电池'], '数据来源': 'KB007'}, ensure_ascii=False, indent=2)
sim_bad = '抱歉，无法提供准确数据。该公司营收约为XXX亿元。'
simulated_outputs = {'maotai': sim_maotai, 'byd': sim_byd, 'bad': sim_bad}

def parse_json_output(raw):
    try: return json.loads(raw.strip()), 'direct'
    except json.JSONDecodeError: pass
    for m in re.findall(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', raw, re.DOTALL):
        try: return json.loads(m), 'regex'
        except json.JSONDecodeError: continue
    return None, 'failed'

print('=== LLM 输出解析演示 ===')
parsed_list = []
for name, raw in simulated_outputs.items():
    result, method = parse_json_output(raw)
    print(f'\n--- {name} --- 解析方法：{method}')
    if result:
        for k, v in result.items(): print(f'  {k}: {v}')
        result['_source'] = name; parsed_list.append(result)
    else: print('解析失败！需人工处理')

if parsed_list:
    df = pd.DataFrame(parsed_list)
    cols = ['公司名称', '报告年度', '营业收入（亿元）', '收入增速', '净利润（亿元）', '净利润增速']
    ok_cols = [c for c in cols if c in df.columns]
    print('\n=== 成功解析的结构化数据 ==='); print(df[ok_cols].to_string(index=False))

## 17.6 情感分析提示模板（Few-shot）

**Few-shot 提示**通过提供示例显著提升金融情感分析准确率。
金融情感需从**投资者视角**判断。

In [ ]:
# Cell 8: 情感分析 Few-shot 提示演示
test_news = [
    '宁德时代宣布与大众汽车签署长期供货协议，订单金额超500亿元',
    '某头部房企债务违约规模扩大至300亿元',
    '央行宣布下调存款准备金率0.25个百分点，释放流动性约6000亿元',
    '某科技公司CEO因个人原因辞职，股价盘前下跌8%',
    'A股市场全天成交额突破2万亿元，创年内新高',
    '某白酒龙头公司上调明年出货计划',
]
sim_results = [
    {'情感': '正面', '强度': 0.85}, {'情感': '负面', '强度': -0.90},
    {'情感': '正面', '强度': 0.65}, {'情感': '负面', '强度': -0.75},
    {'情感': '正面', '强度': 0.55}, {'情感': '正面', '强度': 0.60},
]

print('=== 批量情感分析结果（模拟 LLM 输出）===')
df_sent = pd.DataFrame({'新闻': [n[:25]+'...' for n in test_news],
    '情感': [r['情感'] for r in sim_results],
    '情感强度': [r['强度'] for r in sim_results]})
print(df_sent.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
strengths = [r['强度'] for r in sim_results]
s_colors = ['#d62728' if s < 0 else '#2ca02c' for s in strengths]
ax.bar(range(len(test_news)), strengths, color=s_colors, alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(range(len(test_news)))
ax.set_xticklabels([n[:18]+'...' for n in test_news], rotation=15, ha='right', fontsize=9)
ax.set_ylabel('情感强度'); ax.set_title('金融新闻情感分析结果（模拟）'); ax.set_ylim(-1.1, 1.1)
plt.tight_layout(); plt.show()
print(f'情绪指数：{np.mean(strengths):+.3f}')

## 17.7 思维链（Chain-of-Thought）提示示例

**思维链（CoT）**提示让模型先逐步分析再下结论，
显著提升准确率并增加可审核性。

In [ ]:
# Cell 9: 思维链提示模板与模拟推理过程
COT_TEMPLATE = (
    '你是一位资深证券分析师，请对以下公司进行财务健康度分析。\n'
    '要求：先逐步分析各项财务指标，最后给出综合结论（不构成投资建议）。\n\n'
    '数据：\n{data}\n\n步骤1(盈利) 步骤2(偿债) 步骤3(成长) 步骤4(风险) 步骤5(综合)'
)
example_data = ('公司：某消费品龙头（虚构示例）\n'
    '营收：120亿（+15%）净利：18亿（+20%）\n'
    '净资产：80亿 总资产：150亿 总负债：70亿 流动资产：45亿 流动负债：30亿'
)
simulated_cot = (
    '步骤1 ROE=18/80=22.5%，净利率=18/120=15% -> 盈利能力良好\n'
    '步骤2 流动比率=45/30=1.5，资产负债率=70/150=46.7% -> 偿债能力良好\n'
    '步骤3 营收+8%->+12%->+15%，利润+5%->+15%->+20% -> 成长性较好\n'
    '步骤4 负债率接近50%，增速持续性待观察\n'
    '步骤5 财务质量较好。注意：本分析不构成投资建议。'
)
print('=== 思维链提示模板 ===')
print(COT_TEMPLATE.format(data=example_data))
print('\n=== 模拟 LLM 推理输出 ===')
print(simulated_cot)

print('\n=== 从 CoT 输出中自动提取数值 ===')
roe_m = re.search(r'ROE=([\d.]+)/([\d.]+)=([\d.]+)', simulated_cot)
cr_m  = re.search(r'流动比率=([\d.]+)/([\d.]+)=([\d.]+)', simulated_cot)
dar_m = re.search(r'资产负债率=([\d.]+)/([\d.]+)=([\d.]+)', simulated_cot)
for k, m in [('ROE (%)', roe_m), ('流动比率', cr_m), ('资产负债率(%)', dar_m)]:
    v = float(m.group(3)) if m else None
    print(f'  {"OK" if v else "MISS"} {k}: {v}')

## 17.8 真实 LLM API 调用示例（离线保护）

展示 OpenAI 兼容接口（适用于 GPT-4、通义千问、DeepSeek 等）。
**用 `try/except` 保护，无 API Key 时优雅跳过。**

In [ ]:
# Cell 10: 真实 LLM API 调用示例（离线保护）
import os
API_CONFIGS = {
    'openai':   {'base_url': 'https://api.openai.com/v1', 'api_key_env': 'OPENAI_API_KEY'},
    'deepseek': {'base_url': 'https://api.deepseek.com/v1', 'api_key_env': 'DEEPSEEK_API_KEY'},
    'qwen':     {'base_url': 'https://dashscope.aliyuncs.com/compatible-mode/v1',
                 'api_key_env': 'DASHSCOPE_API_KEY'},
}

def call_llm_api(prompt, model='gpt-4o-mini', temperature=0.1, max_tokens=500):
    cfg = API_CONFIGS['deepseek'] if 'deepseek' in model.lower() else \
          API_CONFIGS['qwen'] if 'qwen' in model.lower() else API_CONFIGS['openai']
    api_key = os.environ.get(cfg['api_key_env'], '')
    if not api_key:
        raise EnvironmentError(f'未找到 API Key（环境变量 {cfg["api_key_env"]}）')
    try: from openai import OpenAI
    except ImportError: raise ImportError('请安装 openai 库：pip install openai')
    client = OpenAI(api_key=api_key, base_url=cfg['base_url'])
    resp = client.chat.completions.create(
        model=model,
        messages=[{'role': 'system', 'content': '你是一位专业的金融分析助手。'},
                  {'role': 'user', 'content': prompt}],
        temperature=temperature, max_tokens=max_tokens)
    return resp.choices[0].message.content

def rag_answer(question, top_k=3):
    docs = retrieve(question, top_k=top_k, verbose=False)
    prompt = build_rag_prompt(question, docs, task='qa')
    try:
        return call_llm_api(prompt, model='gpt-4o-mini', temperature=0.1)
    except EnvironmentError as e:
        print(f'[跳过真实 API 调用] {e}')
        summary = '\n'.join([f'  [{d["id"]}] {d["text"][:60]}...' for d in docs])
        return f'[模拟回答 - 需配置 API Key]\n检索到 {len(docs)} 条相关片段：\n{summary}'
    except Exception as e:
        return f'[API 调用失败] {type(e).__name__}: {e}'

print('=== RAG 问答流水线演示 ===')
for q in ['茅台2023年营业收入是多少？', '科创板上市需要满足哪些条件？']:
    print(f'\nQ: {q}'); answer = rag_answer(q)
    print(f'A: {answer[:200]}'); print('-'*50)

## 17.9 RAG 检索质量评估

- **Hit@k**：Top-k 结果中是否包含正确答案
- **MRR**：正确答案排名倒数的均值

In [ ]:
# Cell 11: RAG 检索质量评估（MRR + Hit@k）
eval_dataset = [
    ('贵州茅台2023年营业收入增速是多少？', ['KB001']),
    ('央行2024年降准释放了多少流动性？', ['KB002']),
    ('科创板对研发投入比例有何要求？', ['KB003']),
    ('宁德时代2023年海外收入占比是多少？', ['KB004']),
    ('注册制下IPO审核主体是谁？', ['KB005']),
    ('商业银行核心一级资本充足率最低要求？', ['KB006']),
    ('比亚迪2023年新能源汽车销量是多少？', ['KB007']),
    ('投资者适当性分几个风险等级？', ['KB008']),
    ('沪深300的市盈率大概是多少倍？', ['KB009']),
    ('DeepSeek-V3的参数量是多少？', ['KB010']),
]

def evaluate_retrieval(data, ks=None):
    if ks is None: ks = [1, 3, 5]
    hits = {k: [] for k in ks}; rr_list = []
    for q, rel in data:
        ret = retrieve(q, top_k=max(ks), verbose=False)
        rids = [r['id'] for r in ret]
        for k in ks: hits[k].append(int(any(r in rids[:k] for r in rel)))
        rr = next((1.0/rank for rank, rid in enumerate(rids,1) if rid in rel), 0.0)
        rr_list.append(rr)
    m = {'MRR': np.mean(rr_list)}
    for k in ks: m[f'Hit@{k}'] = np.mean(hits[k])
    return m, hits, rr_list

metrics, hit_res, rr_list = evaluate_retrieval(eval_dataset)
print('=== TF-IDF RAG 检索质量评估 ===')
for m, v in metrics.items(): print(f'  {m}: {v:.4f} ({v:.1%})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax1 = axes[0]
m_names, m_vals = list(metrics.keys()), list(metrics.values())
mc_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = ax1.bar(m_names, m_vals, color=mc_colors[:len(m_names)], alpha=0.8)
ax1.set_ylim(0, 1.15); ax1.set_title('TF-IDF RAG 检索评估指标')
for bar, val in zip(bars, m_vals):
    ax1.text(bar.get_x()+bar.get_width()/2, val+0.03, f'{val:.2%}',
             ha='center', fontsize=11, fontweight='bold')
ax2 = axes[1]
rr_cols = ['#2ca02c' if r==1 else '#ff7f0e' if r>0 else '#d62728' for r in rr_list]
ax2.bar(range(len(rr_list)), rr_list, color=rr_cols, alpha=0.8)
ax2.axhline(np.mean(rr_list), color='navy', linestyle='--', lw=2,
            label=f'MRR={np.mean(rr_list):.3f}')
ax2.set_title('逐题检索倒数排名'); ax2.legend()
ax2.set_xticks(range(len(rr_list))); ax2.set_xticklabels([f'Q{i+1}' for i in range(len(rr_list))])
plt.suptitle('RAG 检索质量评估', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('\n深度嵌入模型（BGE-m3）通常可达到 Hit@3 >= 90%，远优于 TF-IDF。')

## 17.10 幻觉风险演示与验证策略

幻觉是 LLM 在金融场景最危险的问题。

In [ ]:
# Cell 12: 幻觉风险演示与验证策略
hall_cases = [
    ('贵州茅台2023年营业收入', '约为1350亿元，增长约12%',
     '1476.94亿元，增长18.04%（KB001）', '低估营收和增速'),
    ('科创板研发投入占比要求', '不低于3%',
     '原则上不低于5%（KB003）', '低估比例要求'),
    ('银行核心一级资本充足率', '最低要求为4%',
     '不低于5%（KB006）', '使用国际标准而非国内要求'),
]
print('=== 幻觉风险演示（无 RAG vs 有 RAG）===\n')
for q, hall, correct, err in hall_cases:
    print(f'问题：{q}')
    print(f'  [错误幻觉]：{hall}')
    print(f'  [RAG正确]：{correct}')
    print(f'  [错误类型]：{err}\n')

strategies = pd.DataFrame([
    {'策略': 'RAG（检索增强生成）', '效果分': 5, '代价分': 4},
    {'策略': '要求引用来源', '效果分': 4, '代价分': 1},
    {'策略': '温度设为0', '效果分': 3, '代价分': 1},
    {'策略': '多次采样投票', '效果分': 3, '代价分': 3},
    {'策略': '人工审核节点', '效果分': 5, '代价分': 5},
    {'策略': '与数据库交叉验证', '效果分': 4, '代价分': 3},
])
print('=== 幻觉防范策略对比 ==='); print(strategies.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
y_pos = range(len(strategies))
ax.barh([i+0.2 for i in y_pos], strategies['效果分'], height=0.35,
        color='#2ca02c', alpha=0.8, label='防幻觉效果（1-5）')
ax.barh([i-0.2 for i in y_pos], strategies['代价分'], height=0.35,
        color='#d62728', alpha=0.6, label='实施代价（1-5）')
ax.set_yticks(y_pos); ax.set_yticklabels(strategies['策略'], fontsize=9)
ax.set_xlabel('评分（1-5）'); ax.set_title('幻觉防范策略：效果 vs 代价')
ax.legend(); ax.set_xlim(0, 6); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 17.11 综合流水线演示

整合本章所有模块，演示完整的**金融 RAG 问答流水线**。

In [ ]:
# Cell 13: 综合流水线演示
import datetime

class FinancialRAGPipeline:
    def __init__(self, kb, vec, dv):
        self.kb = kb; self.vec = vec; self.dv = dv; self.log = []
    def _retrieve(self, query, top_k=3):
        sc = cosine_similarity(self.vec.transform([query]), self.dv)[0]
        idx = sc.argsort()[::-1][:top_k]
        return [{**self.kb[i], 'score': float(sc[i])} for i in idx]
    def answer(self, query, top_k=3):
        docs = self._retrieve(query, top_k)
        ctx = '\n'.join([f'[{d["id"]}] {d["text"][:60]}...' for d in docs])
        ans = '[模拟生成] 基于检索到的参考资料。实际部署时替换为真实 API 调用。'
        self.log.append({'ts': datetime.datetime.now().isoformat()[:19],
                         'query': query, 'ids': [d['id'] for d in docs]})
        return {'query': query, 'answer': ans, 'sources': [d['id'] for d in docs]}

pipeline = FinancialRAGPipeline(knowledge_base, vectorizer, doc_vectors)
demo_qs = ['贵州茅台2023年分红总额是多少？',
           '注册制改革后A股IPO审核流程有什么变化？',
           'DeepSeek的训练成本与其他模型相比有何优势？']
print('=== 金融 RAG 流水线演示 ===')
for q in demo_qs:
    res = pipeline.answer(q, top_k=2)
    print(f'\n问：{res["query"]}')
    print(f'答：{res["answer"]}')
    print(f'来源：{res["sources"]}')

print('\n=== 审计日志（最近3条）===')
audit_df = pd.DataFrame(pipeline.log[-3:])
print(audit_df[['ts', 'query', 'ids']].to_string(index=False))

# 流水线架构可视化
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
steps = [('用户问题','#4C72B0'), ('TF-IDF\n检索','#55A868'), ('Top-k\n片段','#55A868'),
         ('构造\n提示','#C44E52'), ('LLM\n生成','#C44E52'),
         ('JSON\n解析','#8172B2'), ('审核\n输出','#937860')]
for i, (label, color) in enumerate(steps):
    x = i*1.5+0.5
    ax.add_patch(mpatches.FancyBboxPatch((x-0.5,0.2), 0.9, 0.6,
                  boxstyle='round,pad=0.05', facecolor=color,
                  edgecolor='white', linewidth=2, alpha=0.85))
    ax.text(x, 0.5, label, ha='center', va='center', fontsize=9,
            color='white', fontweight='bold')
    if i < len(steps)-1:
        ax.annotate('', xy=(x+1.05,0.5), xytext=(x+0.45,0.5),
                    arrowprops=dict(arrowstyle='->', color='#444', lw=2))
ax.set_xlim(0, len(steps)*1.5); ax.set_ylim(0,1)
ax.set_title('金融 RAG 问答流水线架构', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

---
## 习题参考解答

以下 cell 对应正文习题，可直接运行验证。

In [ ]:
# === 习题 17.1：RAG 三步骤演示 ===
print('习题 17.1：RAG 三步骤演示'); print('='*60)
question = '宁德时代2023年的研发费用是多少？'
print(f'步骤1：分块（知识库共{len(knowledge_base)}条片段，均值',
      f'{np.mean([len(d["text"]) for d in knowledge_base]):.0f}字符/片段）')
print('\n步骤2：检索（TF-IDF 余弦相似度）')
retrieved = retrieve(question, top_k=2, verbose=True)
print('\n步骤3：构造提示 + LLM 生成')
p = build_rag_prompt(question, retrieved, task='qa')
print(f'  提示长度：{len(p)} 字符')
print('  模拟回答：根据宁德时代2023年年报（KB004），公司研发费用为183.49亿元，')
print('           研发人员近2万人，占员工总数约13%。（来源：KB004）')
print('\nRAG缓解的问题：知识截止->实时更新知识库；幻觉->基于原文而非记忆')

In [ ]:
# === 习题 17.2：并购公告信息抽取提示设计 ===
print('习题 17.2：并购公告信息抽取提示模板'); print('='*60)
MA_TEMPLATE = (
    '你是专业的信息抽取系统，请从并购公告中提取结构化信息。\n'
    '按 JSON 格式输出（字段不存在时输出 null）：\n'
    '{{"公告日期":"YYYY-MM-DD", "收购方":"...", "被收购方":"...", '
    '"交易金额_亿元":数字, "交易方式":"现金/股票/混合", '
    '"预期完成时间":"...", "收购比例":"百分比", "置信度":"高/中/低"}}\n\n'
    '公告文本：{text}'
)
test_ann = ('本公司2024年6月8日审议通过收购长三角新能源有限公司70%股权的议案，'
    '以发行股票及现金相结合方式，总对价约15.3亿元，预计2024年年底前完成交割。')
print('提示模板（前350字符）：')
print(MA_TEMPLATE.format(text=test_ann)[:350])
print('\n模拟抽取结果：')
sim = {'公告日期': '2024-06-08', '收购方': '本公司', '被收购方': '长三角新能源有限公司',
       '交易金额_亿元': 15.3, '交易方式': '混合',
       '预期完成时间': '2024年年底前', '收购比例': '70%', '置信度': '高'}
print(json.dumps(sim, ensure_ascii=False, indent=2))

In [ ]:
# === 习题 17.3：替换知识库为 IPO 规则 ===
print('习题 17.3：IPO 规则知识库检索演示'); print('='*60)
ipo_kb = [
    {'id': 'IPO001', 'source': '科创板上市规则（财务标准）',
     'text': ('科创板上市财务标准之一：市值不低于10亿元，近两年净利润为正且累计不低于5000万元。'
              '科技属性：研发投入占营收比例不低于5%。')},
    {'id': 'IPO002', 'source': '注册制改革-审核流程说明',
     'text': ('全面注册制下IPO审核流程：申报->交易所问询->证监会注册->发行。'
              '审核权下放至交易所，时间压缩至平均12-18个月。')},
    {'id': 'IPO003', 'source': '主板上市财务条件',
     'text': '主板注册制：近三年净利润为正且累计不低于1.5亿元，或近三年营收累计不低于15亿元。'},
    {'id': 'IPO004', 'source': '创业板上市条件',
     'text': '创业板：市值不低于10亿，近两年净利润为正且累计不低于5000万元。'},
    {'id': 'IPO005', 'source': 'IPO信息披露要求',
     'text': '注册制信息披露核心：招股书须真实、准确、完整。中介机构承担连带责任。'},
]
ipo_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3), max_features=3000, sublinear_tf=True)
ipo_vecs = ipo_vec.fit_transform([d['text'] for d in ipo_kb])
def retrieve_ipo(q, top_k=3):
    sc = cosine_similarity(ipo_vec.transform([q]), ipo_vecs)[0]
    return [(ipo_kb[i], float(sc[i])) for i in sc.argsort()[::-1][:top_k]]
for q in ['科创板上市需要满足哪些财务条件？', '注册制下 IPO 审核流程是怎样的？']:
    print(f'\n问题：{q}')
    for rank, (doc, score) in enumerate(retrieve_ipo(q, top_k=3), 1):
        print(f'  Rank {rank} [{doc["id"]}] sim={score:.4f} -> {doc["text"][:60]}...')
print('\nTF-IDF依赖关键词；BGE-m3基于语义，能处理同义表达但需模型文件。')

In [ ]:
# === 习题 17.4 & 17.5：风险分析 + 架构设计要点 ===
print('习题 17.4：AI 选股助手风险分析'); print('='*60)
for tag, items in [
    ('技术风险', ['幻觉：可能编造财务数据，导致错误建议',
                 '知识截止：不知道最新行情，建议可能已过时',
                 '个性化缺失：无法理解用户风险偏好']),
    ('合规风险', ['无牌经营：向公众发布投资建议须持证券投资咨询牌照',
                 '适当性违规：AI无法评估投资者风险承受能力',
                 '责任归属：AI建议导致损失，法律责任归属不明']),
    ('改进方案', ['功能限制：仅做信息整理，不给具体买卖建议',
                 '持牌审核：关键输出必须经持牌分析师复核',
                 '免责声明：明确标注非投资建议']),
]:
    print(f'\n[{tag}]：')
    for i, item in enumerate(items, 1): print(f'  {i}. {item}')

print('\n' + '='*60)
print('习题 17.5：私有化 RAG 系统架构要点'); print('='*60)
for key, value in [
    ('数据来源', '内部研报库 / 监管公告数据库 / 公开年报PDF（3+种来源）'),
    ('分块策略', 'PDF按段落分块（512 token），表格单独处理，重叠64 token'),
    ('嵌入模型', 'BGE-m3（开源，中英双语，完全本地运行）'),
    ('向量数据库', 'Milvus（开源，百亿级向量，私有化部署）'),
    ('查询流程', '问题嵌入->ANN检索Top-20->Cross-Encoder重排->Top-5->LLM生成'),
    ('安全控制', 'RBAC权限管理 + 数据隔离 + 审计日志 + 内网隔离'),
    ('成本估算', '租赁4xA100约80万/年 vs 商业API约500万/年（省约80%）'),
    ('主要风险', '嵌入模型对长尾术语覆盖有限；PDF表格解析质量；运维成本'),
]:
    print(f'  [{key}] {value}')

print('\n核心结论：LLM 是强大的信息处理辅助工具，不是决策工具。')
print('在金融领域，最终决策和责任仍需由持牌专业人员承担。')